# Lab 3: Gesture Recognition using Convolutional Neural Networks

In this lab you will train a convolutional neural network to make classifications on different hand gestures. By the end of the lab, you should be able to:

1. Load and split data for training, validation and testing
2. Train a Convolutional Neural Network
3. Apply transfer learning to improve your model

Note that for this lab we will not be providing you with any starter code. You should be able to take the code used in previous labs, tutorials and lectures and modify it accordingly to complete the tasks outlined below.

### What to submit

Submit a PDF file containing all your code, outputs, and write-up
from parts 1-5. You can produce a PDF of your Google Colab file by
going to **File > Print** and then save as PDF. The Colab instructions
has more information. Make sure to review the PDF submission to ensure that your answers are easy to read. Make sure that your text is not cut off at the margins. 

**Do not submit any other files produced by your code.**

Include a link to your colab file in your submission.

Please use Google Colab to complete this assignment. If you want to use Jupyter Notebook, please complete the assignment and upload your Jupyter Notebook file to Google Colab for submission. 

## Colab Link

Include a link to your colab file here

Colab Link: 

## Dataset

American Sign Language (ASL) is a complete, complex language that employs signs made by moving the
hands combined with facial expressions and postures of the body. It is the primary language of many
North Americans who are deaf and is one of several communication options used by people who are deaf or
hard-of-hearing. The hand gestures representing English alphabet are shown below. This lab focuses on classifying a subset
of these hand gesture images using convolutional neural networks. Specifically, given an image of a hand
showing one of the letters A-I, we want to detect which letter is being represented.

![alt text](https://www.disabled-world.com/pics/1/asl-alphabet.jpg)

## Part B. Building a CNN [50 pt]

For this lab, we are not going to give you any starter code. You will be writing a convolutional neural network
from scratch. You are welcome to use any code from previous labs, lectures and tutorials. You should also
write your own code.

You may use the PyTorch documentation freely. You might also find online tutorials helpful. However, all
code that you submit must be your own.

Make sure that your code is vectorized, and does not contain obvious inefficiencies (for example, unecessary
for loops, or unnecessary calls to unsqueeze()). Ensure enough comments are included in the code so that
your TA can understand what you are doing. It is your responsibility to show that you understand what you
write.

**This is much more challenging and time-consuming than the previous labs.** Make sure that you
give yourself plenty of time by starting early.

### 1. Data Loading and Splitting [5 pt]

Download the anonymized data provided on Quercus. To allow you to get a heads start on this project we will provide you with sample data from previous years. Split the data into training, validation, and test sets.

Note: Data splitting is not as trivial in this lab. We want our test set to closely resemble the setting in which
our model will be used. In particular, our test set should contain hands that are never seen in training!

Explain how you split the data, either by describing what you did, or by showing the code that you used.
Justify your choice of splitting strategy. How many training, validation, and test images do you have?

For loading the data, you can use plt.imread as in Lab 1, or any other method that you choose. You may find
torchvision.datasets.ImageFolder helpful. (see https://pytorch.org/docs/stable/torchvision/datasets.html?highlight=image%20folder#torchvision.datasets.ImageFolder
) 

In [3]:
# Load the dataset (paths + labels only) 
import os
import re
import random
from collections import Counter

import torch
import torchvision
from torchvision import datasets, transforms

# Path to the folder that contains the A/ ... I/ subfolders,
# relative to this notebook's folder (the "Lab 3" folder).
DATA_DIR = 'Lab3 Dataset/Lab3_Gestures_Summer'

# ImageFolder uses each subfolder name (A-I) as the class label.
# It only stores (file path, label) pairs here; pixels are read from disk
# lazily, one image at a time, when the dataset is indexed later.
# ToTensor is a generic conversion (pixels -> float tensor in [0, 1]);
# any model-specific preprocessing will be added in later parts.
full_dataset = datasets.ImageFolder(DATA_DIR, transform=transforms.ToTensor())

classes = full_dataset.classes          # ['A', 'B', ..., 'I']
samples = full_dataset.samples          # list of (path, class_index)
print('Classes:', classes)
print('Total images:', len(samples))


Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']
Total images: 2219


In [4]:
# Work out which physical hand each image comes from 
# Every filename looks like  <N>_<letter>_<view>.jpg , e.g. 28_A_2.jpg
# Checking all files shows the number N always follows this rule:
#     N = 27 * hand_id + 3 * letter_index + (view - 1)
# i.e. each contributor's hand owns a block of 27 numbers
# (9 letters x 3 views). So hand_id = N // 27 identifies the same hand
# across ALL letters, not just the 2-3 views of a single letter.
FILENAME_PATTERN = re.compile(r'^(\d+)_([A-I])_(\d+)\.jpg$')
IMAGES_PER_HAND_BLOCK = 27   # 9 letters x 3 views

hand_ids = []   # hand_ids[i] is the hand that image i belongs to
for path, label in samples:
    match = FILENAME_PATTERN.match(os.path.basename(path))
    if match is None:
        raise ValueError(f'Unexpected filename format: {path}')
    number, letter, view = int(match.group(1)), match.group(2), int(match.group(3))

    # Sanity checks: stop if any file breaks the assumed naming rule,
    # rather than silently grouping it wrongly.
    if letter != classes[label]:
        raise ValueError(f'Letter in filename does not match its folder: {path}')
    if number % IMAGES_PER_HAND_BLOCK != 3 * classes.index(letter) + (view - 1):
        raise ValueError(f'Filename does not follow the numbering rule: {path}')

    hand_ids.append(number // IMAGES_PER_HAND_BLOCK)

images_per_hand = Counter(hand_ids)
print('Number of distinct hands:', len(images_per_hand))
print('Letters covered per hand (count of hands):',
      dict(Counter(len({classes[samples[i][1]] for i, h in enumerate(hand_ids) if h == hand})
                   for hand in images_per_hand)))


Number of distinct hands: 88
Letters covered per hand (count of hands): {9: 80, 6: 1, 5: 2, 1: 3, 8: 2}


In [5]:
# Split by HAND (not by individual image) 
SEED = 42                     # fixed seed -> the same split every run
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15   # test gets the remaining ~15%

# Shuffle the list of hands (sorted first so the result doesn't depend
# on file-system ordering).
unique_hands = sorted(images_per_hand)
rng = random.Random(SEED)
rng.shuffle(unique_hands)

# Walk through the shuffled hands and give each WHOLE hand to one split:
# fill training until it reaches 70% of all images, then validation
# until train+val reaches 85%, and put the rest in test.
total = len(samples)
hand_to_split = {}
n_train = n_val = 0
for hand in unique_hands:
    if n_train < TRAIN_FRAC * total:
        hand_to_split[hand] = 'train'
        n_train += images_per_hand[hand]
    elif n_train + n_val < (TRAIN_FRAC + VAL_FRAC) * total:
        hand_to_split[hand] = 'val'
        n_val += images_per_hand[hand]
    else:
        hand_to_split[hand] = 'test'

# Turn the hand-level assignment into image indices for each split.
split_indices = {'train': [], 'val': [], 'test': []}
for idx, hand in enumerate(hand_ids):
    split_indices[hand_to_split[hand]].append(idx)


In [6]:
# Verify the split
hands_in = {name: {hand_ids[i] for i in idx} for name, idx in split_indices.items()}

print('Split   Images   % of total   Hands')
for name, idx in split_indices.items():
    print(f'{name:6s} {len(idx):7d} {100 * len(idx) / total:11.1f}% {len(hands_in[name]):7d}')
print(f'Total  {total:7d}')

print('\nImages per class:')
print('       ' + ''.join(f'{c:>5s}' for c in classes))
for name, idx in split_indices.items():
    counts = Counter(samples[i][1] for i in idx)
    print(f'{name:6s} ' + ''.join(f'{counts[k]:5d}' for k in range(len(classes))))

# Leakage check: the sets of hand IDs must not overlap.
overlaps = {
    'train & val':  hands_in['train'] & hands_in['val'],
    'train & test': hands_in['train'] & hands_in['test'],
    'val & test':   hands_in['val'] & hands_in['test'],
}
for pair, shared in overlaps.items():
    if shared:
        raise RuntimeError(f'Data leakage! Hands shared by {pair}: {sorted(shared)}')

# Every image must be used exactly once.
all_indices = sorted(split_indices['train'] + split_indices['val'] + split_indices['test'])
if all_indices != list(range(total)):
    raise RuntimeError('Some images are missing from, or repeated across, the splits.')

print('\nVerified: train/val/test hand IDs are disjoint and every image is used exactly once.')


Split   Images   % of total   Hands
train     1562        70.4%      62
val        345        15.5%      13
test       312        14.1%      13
Total     2219

Images per class:
           A    B    C    D    E    F    G    H    I
train    180  174  172  172  172  168  177  171  176
val       37   39   39   39   39   35   39   39   39
test      37   34   34   36   36   33   34   34   34

Verified: train/val/test hand IDs are disjoint and every image is used exactly once.


In [7]:
# PyTorch datasets for later parts 
# Subset just stores the indices; images are still loaded lazily from disk.
from torch.utils.data import Subset

train_dataset = Subset(full_dataset, split_indices['train'])
val_dataset   = Subset(full_dataset, split_indices['val'])
test_dataset  = Subset(full_dataset, split_indices['test'])

print('train:', len(train_dataset), ' val:', len(val_dataset), ' test:', len(test_dataset))
image, label = train_dataset[0]
print('Example image tensor shape:', tuple(image.shape), ' label:', classes[label])


train: 1562  val: 345  test: 312
Example image tensor shape: (3, 224, 224)  label: A


**How the data was split**

The images are organized into one folder per letter (A–I), and each file is named `<N>_<letter>_<view>.jpg`. Checking every file showed that the number follows a fixed rule: `N = 27 × hand + 3 × letter_index + (view − 1)`. In other words, each contributor's hand owns a block of 27 consecutive numbers (9 letters × up to 3 views), so `N // 27` identifies the same physical hand across **all** letters, not just the 2–3 views of one letter. The code checks that every filename follows this rule and stops with an error if one doesn't. In total there are 2,219 images from 88 hands, and 80 of those hands contributed all nine letters.

I split the data **by hand** instead of by image. With a fixed seed (42), the 88 hands are shuffled. Whole hands are then added to the training set until it holds about 70% of the images, then to the validation set until train + validation hold about 85%. The remaining hands form the test set. Every image of a given hand, for every letter and every angle, ends up in exactly one split. The code checks that the hand IDs in the three splits do not overlap and raises an error if they do.

If the images were split at random, different angles of the same hand (or the same person's other letters) would appear in both training and test. The model could then score well by recognizing the person's hand, skin tone, or background instead of the gesture, which would make the test accuracy too optimistic. The validation set is kept hand-disjoint for the same reason, so that hyperparameter choices are based on how well the model generalizes to new people.

**Result:**

| Split | Images | % of total | Hands |
|---|---|---|---|
| Training | 1562 | 70.4% | 62 |
| Validation | 345 | 15.5% | 13 |
| Test | 312 | 14.1% | 13 |
| **Total** | **2219** | | **88** |

Because whole hands must stay together, the proportions cannot be exactly 70/15/15; each hand moves about 27 images at once. Most hands contributed all nine letters, so each split stays roughly balanced across classes (about 168–180 training images per letter, 35–39 validation, and 33–37 test).


### 2. Model Building and Sanity Checking [15 pt]

### Part (a) Convolutional Network - 5 pt

Build a convolutional neural network model that takes the (224x224 RGB) image as input, and predicts the gesture
letter. Your model should be a subclass of nn.Module. Explain your choice of neural network architecture: how
many layers did you choose? What types of layers did you use? Were they fully-connected or convolutional?
What about other decisions like pooling layers, activation functions, number of channels / hidden units?

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GestureCNN(nn.Module):
    """
    CNN for classifying 224x224 RGB images of ASL letters A-I (9 classes).

    Structure (same pattern as LargeNet from Lab 2 / Lecture 4, but deeper
    because our images are 224x224 instead of 32x32):
        4 x [ Conv2d (3x3, padding 1) -> ReLU -> MaxPool2d (2x2) ]
        -> flatten -> Linear -> ReLU -> Linear (9 outputs)

    base_channels and hidden_units are arguments so they are easy to change
    during hyperparameter tuning later.
    """
    def __init__(self, base_channels=8, hidden_units=64):
        super(GestureCNN, self).__init__()
        self.name = "cnn"   # used later for checkpoint file names 

        # Number of output channels for each conv layer. The number of channels
        # doubles every layer while the image size halves 
        c1, c2, c3, c4 = base_channels, base_channels * 2, base_channels * 4, base_channels * 8

        # Convolutional layers: nn.Conv2d(in_channels, out_channels, kernel_size, padding)
        # A 3x3 kernel with padding=1 ("half padding") keeps the height and width
        # the same:  o = (i + 2p - k)/s + 1 = (i + 2 - 3)/1 + 1 = i
        self.conv1 = nn.Conv2d(3,  c1, kernel_size=3, padding=1)   # 3 input channels = RGB
        self.conv2 = nn.Conv2d(c1, c2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(c2, c3, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(c3, c4, kernel_size=3, padding=1)

        # One max pooling layer reused after every conv layer.
        # kernel 2, stride 2 halves the height and width:  o = (i - 2)/2 + 1 = i/2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Size of the flattened feature map going into the first fully connected layer.
        # Height/width after the 4 pooling layers: 224 -> 112 -> 56 -> 28 -> 14
        # so the last feature map is [c4, 14, 14]  ->  c4 * 14 * 14 values per image.
        self.flat_size = c4 * 14 * 14     # = 64 * 14 * 14 = 12544 with the defaults

        # Fully connected (classifier) layers
        self.fc1 = nn.Linear(self.flat_size, hidden_units)
        self.fc2 = nn.Linear(hidden_units, 9)   # one output per letter A-I

    def forward(self, x):
        # x: [N, 3, 224, 224]
        x = self.pool(F.relu(self.conv1(x)))   # conv -> [N, c1, 224, 224], pool -> [N, c1, 112, 112]
        x = self.pool(F.relu(self.conv2(x)))   # conv -> [N, c2, 112, 112], pool -> [N, c2, 56, 56]
        x = self.pool(F.relu(self.conv3(x)))   # conv -> [N, c3, 56, 56],   pool -> [N, c3, 28, 28]
        x = self.pool(F.relu(self.conv4(x)))   # conv -> [N, c4, 28, 28],   pool -> [N, c4, 14, 14]
        x = x.view(-1, self.flat_size)         # flatten -> [N, c4*14*14]
        x = F.relu(self.fc1(x))                # -> [N, hidden_units]
        x = self.fc2(x)                        # -> [N, 9]
        # No softmax here: we return raw scores (logits). nn.CrossEntropyLoss
        # applies softmax internally (Lecture 3). To get probabilities later,
        # use F.softmax(output, dim=1).
        return x


In [9]:
# Sanity check: pass a dummy batch through the model (no training).
model = GestureCNN()
dummy = torch.randn(4, 3, 224, 224)          # batch of N=4 random "images"

# Print the shape after every layer to confirm the calculations above.
x = dummy
print('input        ', tuple(x.shape))
for i, conv in enumerate([model.conv1, model.conv2, model.conv3, model.conv4], start=1):
    x = F.relu(conv(x));  print(f'conv{i} + relu ', tuple(x.shape))
    x = model.pool(x);    print(f'pool{i}        ', tuple(x.shape))
x = x.view(-1, model.flat_size);  print('flatten      ', tuple(x.shape))
x = F.relu(model.fc1(x));         print('fc1 + relu   ', tuple(x.shape))
x = model.fc2(x);                 print('fc2 (output) ', tuple(x.shape))

# The full forward pass should give one score per class for each image.
out = model(dummy)
assert out.shape == (4, 9), f'Expected (4, 9), got {tuple(out.shape)}'

# It should also work on a real image from the training set.
image, label = train_dataset[0]
print('Real image output shape:', tuple(model(image.unsqueeze(0)).shape))

num_params = sum(p.numel() for p in model.parameters())
print('Total trainable parameters:', num_params)


input         (4, 3, 224, 224)
conv1 + relu  (4, 8, 224, 224)
pool1         (4, 8, 112, 112)
conv2 + relu  (4, 16, 112, 112)
pool2         (4, 16, 56, 56)
conv3 + relu  (4, 32, 56, 56)
pool3         (4, 32, 28, 28)
conv4 + relu  (4, 64, 28, 28)
pool4         (4, 64, 14, 14)
flatten       (4, 12544)
fc1 + relu    (4, 64)
fc2 (output)  (4, 9)
Real image output shape: (1, 9)
Total trainable parameters: 827993


**Architecture**

| Layer | Purpose | Output shape (N = batch size) |
|---|---|---|
| Input | RGB image | [N, 3, 224, 224] |
| Conv1 (3→8, 3×3, pad 1) + ReLU | low-level features (edges, colour changes) | [N, 8, 224, 224] |
| MaxPool (2×2, stride 2) | halve height/width | [N, 8, 112, 112] |
| Conv2 (8→16, 3×3, pad 1) + ReLU | combine edges into simple shapes | [N, 16, 112, 112] |
| MaxPool | halve height/width | [N, 16, 56, 56] |
| Conv3 (16→32, 3×3, pad 1) + ReLU | larger parts (finger/thumb shapes) | [N, 32, 56, 56] |
| MaxPool | halve height/width | [N, 32, 28, 28] |
| Conv4 (32→64, 3×3, pad 1) + ReLU | whole-hand features | [N, 64, 28, 28] |
| MaxPool | halve height/width | [N, 64, 14, 14] |
| Flatten | turn feature maps into a vector | [N, 12544] (= 64 × 14 × 14) |
| FC1 (12544→64) + ReLU | combine features | [N, 64] |
| FC2 (64→9) | one score (logit) per letter A–I | [N, 9] |

Sizes come from the output-size formula o = ⌊(i + 2p − k)/s⌋ + 1. For each conv: (i + 2 − 3)/1 + 1 = i, so the size stays the same. For each pool: (i − 2)/2 + 1 = i/2, so the size halves. Total trainable parameters: 827,993.

**Explanation**

The network has 6 weight layers: 4 convolutional layers followed by 2 fully connected layers. It follows the same convolution → ReLU → max pooling pattern as LargeNet from Lab 2, with two more convolutional layers because the input here is 224×224 instead of 32×32. Convolutional layers are used for feature extraction because a fully connected layer applied directly to a 3×224×224 image would need a very large number of weights and would ignore the spatial structure of the image. A convolutional filter reuses the same small set of weights at every position in the image.

All convolutions use 3×3 kernels with padding 1. With this padding, each convolution keeps the height and width unchanged, and each 2×2 max pool (stride 2) halves them: 224 → 112 → 56 → 28 → 14. This makes the feature map sizes easy to calculate. Small 3×3 kernels stacked over several layers can still respond to large regions of the image while using few weights, which is the approach used by VGG. Max pooling reduces computation, gives each later layer a wider view of the image, and makes the network less sensitive to small shifts in hand position.

The number of channels doubles at each layer (8 → 16 → 32 → 64) while the height and width halve. Early layers need only a few filters for simple features such as edges, and later layers need more filters to represent more complex combinations of them. Doubling the channels also stops the amount of information from shrinking too quickly as the image gets smaller. The last feature map (64 × 14 × 14 = 12,544 values) is flattened and passed to a fully connected hidden layer with 64 units, then to an output layer with 9 units, one per letter. ReLU is used after every layer except the last because it is simple and avoids the vanishing gradients of sigmoid. The output layer returns raw scores (logits) with no softmax, because `nn.CrossEntropyLoss` applies softmax internally.

The channel counts (8 to 64) and the 64 hidden units are kept small because there are only about 1,560 training images, which makes a larger network more likely to overfit and slower to train on a CPU. 

### Part (b) Training Code - 5 pt

Write code that trains your neural network given some training data. Your training code should make it easy
to tweak the usual hyperparameters, like batch size, learning rate, and the model object itself. Make sure
that you are checkpointing your models from time to time (the frequency is up to you). Explain your choice
of loss function and optimizer.

### Part (c) “Overfit” to a Small Dataset - 5 pt

One way to sanity check our neural network model and training code is to check whether the model is capable
of “overfitting” or “memorizing” a small dataset. A properly constructed CNN with correct training code
should be able to memorize the answers to a small number of images quickly.

Construct a small dataset (e.g. just the images that you have collected). Then show that your model and
training code is capable of memorizing the labels of this small data set.

With a large batch size (e.g. the entire small dataset) and learning rate that is not too high, You should be
able to obtain a 100% training accuracy on that small dataset relatively quickly (within 200 iterations).

### 3. Hyperparameter Search [15 pt]

### Part (a) - 3 pt

List 3 hyperparameters that you think are most worth tuning. Choose at least one hyperparameter related to
the model architecture.

### Part (b) - 5 pt

Tune the hyperparameters you listed in Part (a), trying as many values as you need to until you feel satisfied
that you are getting a good model. Plot the training curve of at least 4 different hyperparameter settings.

### Part (c) - 3 pt
Choose the best model out of all the ones that you have trained. Justify your choice.

### Part (d) - 4 pt
Report the test accuracy of your best model. You should only do this step once and prior to this step you should have only used the training and validation data.

### 4. Transfer Learning [15 pt]
For many image classification tasks, it is generally not a good idea to train a very large deep neural network
model from scratch due to the enormous compute requirements and lack of sufficient amounts of training
data.

One of the better options is to try using an existing model that performs a similar task to the one you need
to solve. This method of utilizing a pre-trained network for other similar tasks is broadly termed **Transfer
Learning**. In this assignment, we will use Transfer Learning to extract features from the hand gesture
images. Then, train a smaller network to use these features as input and classify the hand gestures.

As you have learned from the CNN lecture, convolution layers extract various features from the images which
get utilized by the fully connected layers for correct classification. AlexNet architecture played a pivotal
role in establishing Deep Neural Nets as a go-to tool for image classification problems and we will use an
ImageNet pre-trained AlexNet model to extract features in this assignment.

### Part (a) - 5 pt
Here is the code to load the AlexNet network, with pretrained weights. When you first run the code, PyTorch
will download the pretrained weights from the internet.

In [ ]:
import torchvision.models
alexnet = torchvision.models.alexnet(pretrained=True)

The alexnet model is split up into two components: *alexnet.features* and *alexnet.classifier*. The
first neural network component, *alexnet.features*, is used to compute convolutional features, which are
taken as input in *alexnet.classifier*.

The neural network alexnet.features expects an image tensor of shape Nx3x224x224 as input and it will
output a tensor of shape Nx256x6x6 . (N = batch size).

Compute the AlexNet features for each of your training, validation, and test data. Here is an example code
snippet showing how you can compute the AlexNet features for some images (your actual code might be
different):

In [ ]:
# img = ... a PyTorch tensor with shape [N,3,224,224] containing hand images ...
features = alexnet.features(img)

**Save the computed features**. You will be using these features as input to your neural network in Part
(b), and you do not want to re-compute the features every time. Instead, run *alexnet.features* once for
each image, and save the result.

### Part (b) - 3 pt
Build a convolutional neural network model that takes as input these AlexNet features, and makes a
prediction. Your model should be a subclass of nn.Module.

Explain your choice of neural network architecture: how many layers did you choose? What types of layers
did you use: fully-connected or convolutional? What about other decisions like pooling layers, activation
functions, number of channels / hidden units in each layer?

Here is an example of how your model may be called:

In [ ]:
# features = ... load precomputed alexnet.features(img) ...
output = model(features)
prob = F.softmax(output)

### Part (c) - 5 pt
Train your new network, including any hyperparameter tuning. Plot and submit the training curve of your
best model only.

Note: Depending on how you are caching (saving) your AlexNet features, PyTorch might still be tracking
updates to the **AlexNet weights**, which we are not tuning. One workaround is to convert your AlexNet
feature tensor into a numpy array, and then back into a PyTorch tensor.

In [ ]:
tensor = torch.from_numpy(tensor.detach().numpy())

### Part (d) - 2 pt
Report the test accuracy of your best model. How does the test accuracy compare to Part 3(d) without transfer learning?